<a href="https://colab.research.google.com/github/DaniJonesOcean/etc-impacts-great-lakes/blob/main/notebooks/cartopy/fig4_genesis_maps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cartopy shapely pyproj

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [3]:
# Mounting Google Drive in Google Colab
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [4]:
myPath = "/content/gdrive/MyDrive/ETC Clustering Paper/cfsr_storms_labeled_k2.csv"
df = pd.read_csv(myPath)
df.head()

,min_p_cent,max_p_grad,max_radius,max_uv,fraction_of_time_in_GLR,maturity_glr0_minus_genesis_ratio,sup_ttl_precip,mi_ttl_precip,huron_ttl_precip,erie_ttl_precip,...,enso_best_value,ipo_value,nao_value,pdo_value,pna_value,pol_value,qbo_value,wp_value,k2_cluster,k2_posterior
0,1007.534375,13.014696,888.486645,132.679247,0.200000,0.533333,0.008939,0.004745,0.006958,0.007487,...,0.224352,0.128273,-2.023275,-0.43032,-1.399966,0.670369,5.300417,1.403931,0,0.999535
1,1004.919297,14.567884,743.147127,127.950766,0.000000,0.266667,0.007820,0.003807,0.005326,0.008860,...,0.224352,0.128273,-2.023275,-0.43032,-1.399966,0.670369,5.300417,1.403931,1,0.991803
2,984.576641,24.076204,1623.945014,107.086910,0.166667,0.125000,0.007253,0.031011,0.027484,0.017373,...,0.224352,0.128273,-2.023275,-0.43032,-1.399966,0.670369,5.300417,1.403931,1,0.999581
3,1008.355938,17.208477,1660.529852,117.163265,0.111111,0.000000,0.010575,0.010530,0.012207,0.008459,...,0.224352,0.128273,-2.023275,-0.43032,-1.399966,0.670369,5.300417,1.403931,1,0.999998
4,1005.740703,12.624121,590.380661,90.981127,0.500000,0.500000,0.007816,0.009645,0.007444,0.016935,...,0.224352,0.128273,-2.023275,-0.43032,-1.399966,0.670369,5.300417,1.403931,0,1.000000


In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ----------------------------
# Harmonized naming + colors
# ----------------------------
TYPE1 = "Type 1 (early-life-cycle)"
TYPE2 = "Type 2 (late-life-cycle)"
CLUSTER_TO_TYPE = {0: TYPE2, 1: TYPE1}  # edit if needed

COLORS = {TYPE1: "tab:blue", TYPE2: "tab:orange"}
MARKER = "x"

if "storm_type" not in df.columns:
    df["storm_type"] = df["k2_cluster"].map(CLUSTER_TO_TYPE)

# Make sure that season exists
assert {"lon_gen", "lat_gen", "storm_type", "season"}.issubset(df.columns)

# ----------------------------
# Map settings
# ----------------------------
lon_min, lon_max = -130, -66
lat_min, lat_max = 24, 65

def setup_ax(ax):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linewidth=0.6)
    ax.add_feature(cfeature.LAND, alpha=0.3)
    ax.add_feature(cfeature.OCEAN, alpha=0.4)
    ax.add_feature(cfeature.LAKES, alpha=0.6)
    ax.add_feature(cfeature.RIVERS, linewidth=0.4)

    gl = ax.gridlines(
        draw_labels=True, dms=True, x_inline=False, y_inline=False,
        color="k", alpha=0.35, linestyle="--", linewidth=0.6
    )
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {"size": 11}
    gl.ylabel_style = {"size": 11}

def smooth2d(H, iters=2):
    K = np.array([[1, 2, 1],
                  [2, 4, 2],
                  [1, 2, 1]], dtype=float)
    K /= K.sum()

    A = H.astype(float)
    for _ in range(iters):
        P = np.pad(A, 1, mode="edge")
        A = (
            K[0,0]*P[:-2,:-2] + K[0,1]*P[:-2,1:-1] + K[0,2]*P[:-2,2:] +
            K[1,0]*P[1:-1,:-2] + K[1,1]*P[1:-1,1:-1] + K[1,2]*P[1:-1,2:] +
            K[2,0]*P[2:,:-2] + K[2,1]*P[2:,1:-1] + K[2,2]*P[2:,2:]
        )
    return A

def plot_origin_points_and_density(
    df_in,
    season=None,
    bins=90,
    smooth_iters=2,
    clip_q=0.98,
    outdir="/content/gdrive/MyDrive/ETC Clustering Paper/figures",
    save=True,
):
    os.makedirs(outdir, exist_ok=True)

    if season is not None:
        ddf = df_in[df_in["season"] == season].copy()
        season_str = str(season)
    else:
        ddf = df_in.copy()
        season_str = "ALL"

    fig, axes = plt.subplots(
        2, 2, figsize=(20, 12),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )

    for r, stype in enumerate([TYPE1, TYPE2]):
        d = ddf[ddf["storm_type"] == stype].dropna(subset=["lon_gen", "lat_gen"])

        # ---- Left: points ----
        ax = axes[r, 0]
        setup_ax(ax)
        ax.scatter(
            d["lon_gen"], d["lat_gen"],
            s=18, c=COLORS[stype], alpha=0.35,
            marker=MARKER, linewidths=0.6
        )
        ax.set_title(f"{season_str} — {stype}: Genesis locations (n={len(d)})", fontsize=14)

        # ---- Right: density ----
        ax = axes[r, 1]
        setup_ax(ax)

        H, xedges, yedges = np.histogram2d(
            d["lon_gen"], d["lat_gen"],
            bins=[bins, bins],
            range=[[lon_min, lon_max], [lat_min, lat_max]]
        )
        Hs = smooth2d(H.T, iters=smooth_iters)

        # Clip to prevent Rockies dominating the full range
        if np.any(Hs > 0):
            upper = np.quantile(Hs[Hs > 0], clip_q)
            Hplot = np.clip(Hs, 0, upper)
        else:
            Hplot = Hs

        X, Y = np.meshgrid(xedges, yedges)

        pm = ax.pcolormesh(
            X, Y, Hplot,
            transform=ccrs.PlateCarree(),
            shading="auto",
            alpha=0.9
        )

        # Contours for structure (on the clipped field)
        if np.any(Hplot > 0):
            levels = np.quantile(Hplot[Hplot > 0], [0.6, 0.75, 0.9, 0.97])
            levels = np.unique(levels[np.isfinite(levels)])
            if len(levels) > 1:
                ax.contour(
                    (xedges[:-1] + xedges[1:]) / 2,
                    (yedges[:-1] + yedges[1:]) / 2,
                    Hplot,
                    levels=levels,
                    transform=ccrs.PlateCarree(),
                    linewidths=0.8,
                    alpha=0.8
                )

        ax.set_title(f"{season_str} — {stype}: Genesis density (clipped @ {int(clip_q*100)}th)", fontsize=14)

        cbar = fig.colorbar(pm, ax=ax, orientation="vertical", shrink=0.85, pad=0.02)
        cbar.set_label("Relative count (smoothed, clipped)", fontsize=11)

    fig.suptitle(f"Storm genesis by type — {season_str}", fontsize=18, y=0.98)
    fig.tight_layout()

    if save:
        fname = f"fig4_genesis_points_density_{season_str}.png"
        fig.savefig(os.path.join(outdir, fname), dpi=300, bbox_inches="tight")
        plt.close(fig)
        return fname
    else:
        return fig, axes

# ----------------------------
# Make one per season (+ optional ALL)
# ----------------------------
for s in ["DJF", "MAM", "JJA", "SON"]:
    outname = plot_origin_points_and_density(df, season=s, save=True)
    print("Saved:", outname)

# Optional: all-seasons version (if you want it too)
outname = plot_origin_points_and_density(df, season=None, save=True)
print("Saved:", outname)


Saved: fig4_genesis_points_density_DJF.png
Saved: fig4_genesis_points_density_MAM.png
Saved: fig4_genesis_points_density_JJA.png
Saved: fig4_genesis_points_density_SON.png
Saved: fig4_genesis_points_density_ALL.png
